# Context of the project :
The goal of this project is to create a `supervised classifier IA model` who can predict if a data is in a class or not. The choice of the dataset was free (*as long as the data come from the medical field*), so we choose to work with a dataset of **X-rays images of fractured and not fractured hand and wrist**.

> `Supervised learning `is a type of machine learning where the model is trained on a labeled dataset, meaning that each input data point is associated with a corresponding output label. The model learns to map the input data to the correct output labels during the training process, allowing it to make predictions on new, unseen data.

> `Classification` is a type of supervised learning where the goal is to predict the class label of a given input based on its features. *In this case, the model will learn to classify X-ray images as either "fractured" or "not fractured" based on the patterns and features present in the images.*

## The dataset :
**10 000 images of X-rays images of fractured and not fractured hand and wrist**

Why this dataset ? Because it is a good example of a binary classification problem, and it is a real-world problem that can have a significant impact on the medical field. The model can help doctors to quickly and accurately diagnose fractures, which can lead to better patient outcomes. *Furthermore, some of us are familiar with this type of injury.*

**Sources :**
> https://www.kaggle.com/datasets/osamajalilhassan/bone-fracture-dataset/data



- 2 folders : one with fractured hand and wrist, and one with not fractured hand and wrist
- Each folder contains images of hand and wrist X-rays, with the same format (jpg)
- The dataset is balanced, with the same number of images in each folder

**How to detect fractures ?**

> By definition, a fracture is a break in a bone. On an X-ray, it is necessary to detect abnormalities in the bone, look for fracture lines, check the continuity of the bone and/or look for bone displacement.


## 1. Dataset Exploration and Visualization

## 2. Data Preprocessing

**Import libraries `numpy`, `os` and `sklean.pipeline` to preprocess the images**

In [1]:
import os
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from CvrieLib.preprocessing import image_to_array

**Import our own library `CvrieLib` to preprocess the images, and reload it to be sure to have the last version every time we run the code**

- *The* preprocess_image *function will take an image path and a target size, and will return a preprocessed image ready to be used in the model.*

- *The* flatten_image_array *function will take a 3D array of images and will flatten it into a 2D array, where each row corresponds to an image and each column corresponds to a pixel value.*

In [2]:
import importlib
import CvrieLib.preprocessing as prep

importlib.reload(prep)
# preprocess_image = prep.preprocess_image __nor more used, replaced by a pipeline with scikit learn__
flatten_image_array = prep.flatten_image_array
image_to_array = prep.image_to_array
divide_gray_scale = prep.divide_gray_scale
resize_image = prep.resize_image
normalize_image = prep.normalize_image
bilinear_interpolation = prep.bilinear_interpolation

**Initialisation of variables**

- `X` will be the dataset of preprocessed images, and `Y` will be the array of labels (1 for fractured, 0 for not fractured)
- `img_extension` will be the extension of the images in the dataset, to filter the files in the folders
- `img_size` will be the target size for the images after preprocessing, to ensure that all images have the same dimensions and can be used in the model. *We choose 224x224 because it is a common size for image classification models, and it allows us to capture enough details in the images while keeping the computational cost manageable.*

In [3]:
X = []
Y = []
img_extension = '.jpg'
img_size = [224, 224]

**Create a function to build a pipeline of preprocessing steps for the images and a function to load the dataset from the folders, applying the pipeline to each image.**

**Why a pipeline ?**

> Initially, data preprocessing was done via the preprocess_images function. But with the addition of new preprocessing steps, the function became more complex and less modular. By using a pipeline, we can easily add, remove or modify preprocessing steps without affecting the overall structure of the code.
> This also facilitates testing on the pipeline and allows for faster execution. A pipeline enables the exact same processing to be applied to the training set and, later, to new images that the model has never seen, ensuring that the model is tested under real-world conditions.

`Note: We use FunctionTransformers to allow the pipeline to work with our own functions. We also use kw_args because the pipeline is only designed for one argument in the steps; this tool allows us to specify that for resize, the bilinear interpolation function also takes on a shape.`


In [4]:
def build_image_pipeline(target_size=(img_size[0], img_size[1])):
    return Pipeline([
        ('to_array', FunctionTransformer(image_to_array)),
        ('grayscale', FunctionTransformer(divide_gray_scale)),
        ('resize', FunctionTransformer(
            bilinear_interpolation,
            kw_args={'new_shape': target_size}
        )),
        ('normalize', FunctionTransformer(normalize_image))
    ])

def load_fracture_dataset(directory, label_value, target_size=(img_size[0], img_size[1])):
    files = [f for f in os.listdir(directory) if f.lower().endswith(img_extension)]
    n_files = len(files)
    print(f"Loading {n_files} files from {directory} with label {label_value}")
    data = np.zeros((n_files, target_size[0], target_size[1]), dtype=np.float32)
    labels = np.full(n_files, label_value, dtype=np.int8)
    pipeline = build_image_pipeline(target_size=target_size)
    for i, f in enumerate(files):
        full_path = os.path.join(directory, f)
        try:
            data[i] = pipeline.fit_transform(full_path)
            # data[i] = preprocess_image(full_path, target_size=target_size) __last version__
        except Exception as e:
            print(f"Erros in the {f}: {e}")
    return data, labels


**Load the dataset from the folders, applying the pipeline to each image. We will have two arrays: one for the fractured images and one for the not fractured images, with their corresponding labels.**

Once the dataset is loaded in X and y, we will concatenate the two arrays to have a single dataset and a single array of labels. We will also print the shape of X before flattening to check that the images are correctly loaded and preprocessed.

In [5]:
X_fractured, y_fractured = load_fracture_dataset('Dataset/fractured/', label_value=1)
X_not_fractured, y_not_fractured = load_fracture_dataset('Dataset/not_fractured/', label_value=0)

X = np.concatenate([X_fractured, X_not_fractured], axis=0)
y = np.concatenate([y_fractured, y_not_fractured], axis=0)

print(f"Structure of X after flatten : {X.shape}")

Loading 4840 files from Dataset/fractured/ with label 1
Loading 4623 files from Dataset/not_fractured/ with label 0
Structure of X after flatten : (9463, 224, 224)


**Flatten the images in X to have a 2D array where each row corresponds to an image and each column corresponds to a pixel value. This is necessary for the model to be able to process the data.**

We will also print the shape of X after flattening to check that the images are correctly flattened.

In [6]:
X_flat = flatten_image_array(X)

print(f"Structure of X after flatten : {X_flat.shape}")

Structure of X after flatten : (9463, 50176)



## 3. Model Training and Evaluation

**Import the necessary libraries from `scikit-learn` to train and evaluate the model.**

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

**Split dataset in 80/20, 80% to train and 20% to test, shuffled equally with stratify**

The The segmentation can then be optimized with cross-validation.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=5, stratify=y)

### GridSearch on the dataset to find the best model to use

In [9]:
# param_grid_rf = {
#     'n_estimators': [100, 300, 500],
#     'max_depth': [10, 20, None],
#     'min_samples_split': [2, 5, 10],
#     'class_weight': ['balanced', None]
# }
#
# grid_rf = GridSearchCV(RandomForestClassifier(random_state=0), param_grid_rf, cv=5, scoring='f1', n_jobs=4)
#
# grid_rf.fit(X_train, y_train)
#
# print(f"Best RF Params: {grid_rf.best_params_}")
# print(f"Best RF F1-Score: {grid_rf.best_score_}")
#
# best_model = grid_rf.best_estimator_

In [10]:
# y_prediction = best_model.predict(X_test)
# print(confusion_matrix(y_test, y_prediction))
# print(classification_report(y_test, y_prediction))

### Train on best Model and gives scores

In [11]:
model = RandomForestClassifier(n_estimators=50, max_depth=5, n_jobs=-1, min_samples_split=2, min_samples_leaf=1)

# puis faire du gradient free routines (soit celui de scikit learn soit notre propre)
# pour choisir les hyperparameters du modele

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("Accuracy Score:", accuracy)
print(f"Fractured samples: {len(y_fractured)}")
print(f"Not fractured samples: {len(y_not_fractured)}")

# tester le model sur une image spécifique

Accuracy Score: 0.8235604860010565
Fractured samples: 4840
Not fractured samples: 4623
